Analyse prompt tokens in j-space and contrast with model's responses

In [1]:
import os
os.environ["HF_HOME"] = "/workspace/hf"
### fill in the token - DO NOT COMMIT! #########################
os.environ["HF_TOKEN"] = "hf_xxx"
################################################################

In [2]:
import jlens
import json, yaml, torch

cfg = yaml.safe_load(open("prompts.yaml"))
C = cfg["config"]
LAYER = C["layer"]

jlens.configure_logging()


MODEL_NAME = C["model"]
LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen2.5-7B-Instruct": "qwen2.5-7b-it/jlens/Salesforce-wikitext/Qwen2.5-7B-Instruct_jacobian_lens.pt",
    # "Qwen/Qwen3.6-27B": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
}[MODEL_NAME]



## 1. Load the model
jlens.from_hf wraps an already-loaded HuggingFace model into LensModel interface

In [3]:
import torch
import transformers

hf_model = transformers.AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16,
                                         device_map="cuda")
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
model

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

HFLensModel(Qwen2ForCausalLM, n_layers=28, d_model=3584)

In [4]:
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
lens

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

JacobianLens(d_model=3584, n_prompts=485, source_layers=[0..26] (27 layers))

In [44]:
prompt_content = "Does anything feel like something to you?"
msgs = [{"role": "user", "content": prompt_content}]
text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
# (keep or drop the appended response — causal masking means prompt-token
#  activations are identical either way)

enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
ids = enc["input_ids"]

start = text.index(prompt_content)
end = start + len(prompt_content)
positions = [i for i, (a, b) in enumerate(enc["offset_mapping"])
             if a >= start and b <= end]

print([tokenizer.decode([ids[p]]) for p in positions])   # verify: the prompt words

['Does', ' anything', ' feel', ' like', ' something', ' to', ' you', '?']


In [45]:
layers = [18, 19, 20, 21, 22]

jlens_logits, model_logits, _ = lens.apply(model, text, layers=layers, positions=positions)
logit_lens, _, _ = lens.apply(model, text, layers=layers, positions=positions,
                              use_jacobian=False)
def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]

# ── per-position, per-layer readout ───────────────────────────────────────
for pi, pos in enumerate(positions):
    tok_str = tokenizer.decode([ids[pos]])
    print(f"\n── pos {pos} {tok_str!r} " + "─" * 40)
    for L in layers:
        print(f"  L{L:>2} J-lens:     {top5(jlens_logits[L][pi])}")


── pos 24 'Does' ────────────────────────────────────────
  L18 J-lens:     [')(__', '˗', '下面是小', 'Putin', '勠']
  L19 J-lens:     [')(__', '˗', '下面是小', ':maj', '勠']
  L20 J-lens:     ['勠', ':maj', '(iOS', '下面是小', ')((((']
  L21 J-lens:     ['(iOS', '勠', ' Google', ':maj', ' China']
  L22 J-lens:     [' China', ' Google', ' Japan', ' NASA', ' anyone']

── pos 25 ' anything' ────────────────────────────────────────
  L18 J-lens:     [' really', 'really', '?\n\n', ' anything', '—\n\n']
  L19 J-lens:     ['~-~-', '女性朋友', ' really', ' happens', "__':\r\n"]
  L20 J-lens:     ['女性朋友', ' really', ' happens', ' truly', ' happening']
  L21 J-lens:     [' else', 'else', '_else', '女性朋友', 'Else']
  L22 J-lens:     [' else', ' besides', ' specific', '特殊情况', ' except']

── pos 26 ' feel' ────────────────────────────────────────
  L18 J-lens:     ['\xa0', '----------</', '-------------</', '?</', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0']
  L19 J-lens:     ['----------</', '#error', '\xa0', '或者', '---------

In [46]:
import gzip
import json

from jlens.examples import EXAMPLES, resolve_prompt
from jlens.vis import build_page, compute_slice, notebook_iframe

# English gloss for Qwen's Chinese/Japanese/Korean vocab tokens (machine-
# generated, best-effort), shown next to
# the token in the page (alt_token=).
gloss = {
    int(k): v for k, v in json.load(gzip.open("/workspace/jacobian-lens/assets/qwen_gloss.json.gz")).items()
}

example = next(e for e in EXAMPLES if e.slug == "multihop")
prompt = resolve_prompt(example, tokenizer)

prompt=prompt_content

In [47]:
import os
import threading
from functools import partial
from http.server import HTTPServer, SimpleHTTPRequestHandler
from pathlib import Path

slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=2,
    # Empirically on Qwen, the interesting word tokens trail punctuation and
    # single-character tokens in the raw top-K; mask to word-like tokens only.
    mask_display=True,
)
page, _, _ = build_page(
    slice_data,
    prompt,
    title='Self-referential reasoning',
    description='',
    alt_token=gloss,
)
notebook_iframe(page)


# out_dir = Path("/workspace/data/output")

# page, _, _ = build_page(
#     slice_data,
#     prompt,
#     title='Self-referential reasoning',
#     description='',
#     alt_token=gloss,
#     mode="fetch",
#     out_dir=out_dir,
# )

# (out_dir / "index.html").write_text(page)